<a href="https://colab.research.google.com/github/buiduchuy28082005-lgtm/Financial-Distress-Prediction-Model/blob/master/FD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# PHẦN 1: ĐỌC & HIỂU DỮ LIỆU

## 1.1 Đọc dữ liệu

In [ ]:
!pip install imbalanced-learn seaborn openpyxl

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print('Import thư viện thành công.')

Import thư viện thành công.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Data có thêm SIZE = ln(Total Assets / CPI) và SECTOR (GICS Level 1)
DATA_PATH = '/content/drive/MyDrive/FDMT/Data_Final.csv'
df_raw = pd.read_csv(DATA_PATH)

# Loại bỏ cột tham chiếu (không dùng cho model)
df_raw = df_raw.drop(columns=['total_assets_used'])
df_raw['Date'] = pd.to_datetime(df_raw['Date'])

print(f'Kích thước: {df_raw.shape[0]:,} dòng x {df_raw.shape[1]} cột')
print(f'Năm báo cáo: {df_raw["yearReport"].min()} - {df_raw["yearReport"].max()}')
print(f'Số công ty:  {df_raw["symbol"].nunique()}')
df_raw.head()

---
## 1.2 Hiểu dữ liệu (Data Understanding)

In [ ]:
df_raw.info()

### Danh sách features

In [ ]:
# 32 tỷ số tài chính + SIZE + SECTOR
feature_cols = [col for col in df_raw.columns if col.startswith('N')]
extra_features = ['SIZE', 'SECTOR']
all_features = feature_cols + extra_features

print(f'Tổng số features: {len(all_features)}')
print(f'  - Tỷ số tài chính (N1–N32): {len(feature_cols)} biến')
print(f'  - SIZE:   ln(Total Assets / CPI)')
print(f'  - SECTOR: GICS Level 1, {df_raw["SECTOR"].nunique()} ngành')

print(f'\nDanh sách features:')
print(all_features)

### Thống kê mô tả (Dữ liệu thô)

In [ ]:
# Thống kê mô tả cho các biến số (financial ratios + SIZE)
numeric_features = feature_cols + ['SIZE']
df_raw[numeric_features].describe().T.round(4)

In [ ]:
# Phân phối SECTOR (biến categorical)
print('Phân phối ngành:')
print(df_raw['SECTOR'].value_counts().to_string())

### Kiểm tra missing value

In [ ]:
# Tính tỉ lệ missing của toàn bộ features
missing_df = pd.DataFrame({
    'So luong thieu': df_raw[all_features].isnull().sum(),
    'Ti le (%)': (df_raw[all_features].isnull().sum() / len(df_raw) * 100).round(2)
})
missing_df = missing_df[missing_df['So luong thieu'] > 0].sort_values(
    'Ti le (%)', ascending=False)

print('Các biến bị thiếu giá trị:')
print(missing_df.to_string())

# Visualize
fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(missing_df.index, missing_df['Ti le (%)'],
               color='#E74C3C', alpha=0.8)
ax.set_xlabel('Tỉ lệ missing (%)', fontsize=11)
ax.set_title('Biểu đồ tỉ lệ giá trị thiếu theo biến', fontsize=13)
ax.tick_params(axis='y', labelsize=8)
for bar, val in zip(bars, missing_df['Ti le (%)']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=8)
plt.tight_layout()
plt.show()

N16_NOCREDINT thiếu 100% → sẽ drop. Các biến tăng trưởng N27–N32 thiếu ~10% (do tính YoY nên 4 quý đầu mỗi công ty không có giá trị). SIZE thiếu ~1.6% (277 dòng). Các biến còn lại thiếu < 5% – có thể impute an toàn.

### Kiểm tra Outliers

In [ ]:
# Demo boxplot trên 4 biến đại diện
sample_vars = ['N1_EBITTA', 'N4_RETA', 'N10_CACL', 'N24_TLTA']

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for i, var in enumerate(sample_vars):
    axes[i].boxplot(df_raw[var].dropna(), vert=True, patch_artist=True,
                    boxprops=dict(facecolor='#AED6F1'))
    axes[i].set_title(var, fontsize=10)
    axes[i].set_ylabel('Giá trị' if i == 0 else '')

plt.suptitle('Phân phối các biến chính (dữ liệu thô) – có outliers cực đoan',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Kiểm tra outliers có phải là KQTC firms không
for var in ['N1_EBITTA', 'N4_RETA', 'N10_CACL', 'N24_TLTA']:
    q01 = df_raw[var].quantile(0.01)
    q99 = df_raw[var].quantile(0.99)
    outliers_mask = (df_raw[var] < q01) | (df_raw[var] > q99)
    pct_kqtc_outliers = df_raw.loc[outliers_mask, 'distress_flag'].mean() * 100
    pct_kqtc_normal = df_raw.loc[~outliers_mask, 'distress_flag'].mean() * 100
    print(f'{var}:')
    print(f'  Outliers (<1% hoặc >99%): {pct_kqtc_outliers:.1f}% là KQTC')
    print(f'  Normal range:             {pct_kqtc_normal:.1f}% là KQTC')

## Nhận xét về Outliers:

Kết quả kiểm chứng cho thấy **outliers ở các biến tài chính chính là tín hiệu của KQTC**, không phải nhiễu cần loại bỏ:

| Biến | Tỉ lệ KQTC ở Outliers | Tỉ lệ KQTC ở Normal range | Chênh lệch |
|---|---|---|---|
| N1_EBITTA  | 50.3% | 22.7% | **2.2x** |
| N4_RETA    | 44.3% | 22.8% | **1.9x** |
| N10_CACL   | 49.4% | 22.7% | **2.2x** |
| N24_TLTA   | 55.7% | 22.6% | **2.5x** |

Cả 4 biến đều cho thấy: ở vùng đuôi phân phối (1% hoặc 99%), tỉ lệ công ty KQTC cao gấp 2–2.5 lần so với vùng trung tâm. Điều này khẳng định:

1. **Outliers chứa thông tin dự báo quan trọng** – công ty có chỉ số tài chính cực đoan thường là công ty đang/sắp kiệt quệ tài chính (EBIT âm sâu, lỗ lũy kế lớn, đòn bẩy cực cao). Loại bỏ chúng = loại bỏ chính nhóm cần dự báo.

2. **Nên dùng Winsorize để xử lý**, vì:
   - Giữ lại toàn bộ quan sát (không giảm sample size)
   - Giới hạn ảnh hưởng quá mức của giá trị cực đoan đến hàm loss, đặc biệt với các model nhạy với scale như Logistic Regression và SVM
   - Bảo toàn ordering thông tin: công ty có chỉ số cực đoan vẫn ở đuôi phân phối sau khi clip

=> Áp dụng Winsorize [1%, 99%] trong Phần 4: Xây dựng mô hình, fit chỉ trên train set để tránh data leakage.

---
# PHẦN 2: TIỀN XỬ LÝ

## Drop missing

In [ ]:
df = df_raw.copy()

# Drop cột missing > 80% – check trên TOÀN BỘ features
drop_threshold = 0.8
missing_rate = df[all_features].isnull().mean()
cols_to_drop = missing_rate[missing_rate > drop_threshold].index.tolist()
print(f'Loai bo {len(cols_to_drop)} cot missing > {drop_threshold*100:.0f}%: '
      f'{cols_to_drop}')
df = df.drop(columns=cols_to_drop)

# Cập nhật cả feature_cols và all_features sau khi drop
feature_cols = [col for col in feature_cols if col not in cols_to_drop]
extra_features = [col for col in extra_features if col not in cols_to_drop]
all_features = feature_cols + extra_features

print(f'So feature N* con lai: {len(feature_cols)}')
print(f'Tong so features: {len(all_features)}')

## Fill missing
Điền giá trị thiếu bằng **median của chính công ty đó**. Giúp:
- Bảo toàn đặc thù từng công ty (firm-specific characteristics)
- Không leak thông tin toàn cục (không dùng median toàn bộ data)
- Một số dòng vẫn còn NaN nếu công ty thiếu toàn bộ giá trị 1 biến → sẽ xử lý ở phần sau

In [ ]:
# Fill missing cho các biến số (financial ratios + SIZE) bằng median theo symbol
numeric_feats = feature_cols + ['SIZE']
df[numeric_feats] = (
    df.groupby('symbol')[numeric_feats]
    .transform(lambda x: x.fillna(x.median()))
)

remaining = df[numeric_feats].isnull().sum()
remaining = remaining[remaining > 0]
print(f'Missing sau entity-level imputation: {df[numeric_feats].isnull().sum().sum()}')
if len(remaining) > 0:
    print(f'\nCác cột còn missing (sẽ fill bằng train-median trong Pipeline):')
    print(remaining.to_string())

Missing sau entity-level imputation: 1110

Các cột còn missing (sẽ fill bằng train-median trong Pipeline):
N5_NITS          7
N6_EBITDATA     13
N7_EBITTS        7
N9_EBITDATL     13
N19_WCTS         7
N20_CLTS         7
N27_GNI        160
N28_GTA        192
N29_GOI        160
N30_GRTA       192
N31_GSE        192
N32_GTS        160


In [ ]:
print('=== SO SÁNH TRƯỚC VÀ SAU TIỀN XỬ LÝ ===')
print(f'{"Chỉ tiêu":<35} {"Trước":>12} {"Sau":>12}')
print('-' * 60)
print(f'{"Số dòng":<35} {df_raw.shape[0]:>12,} {df.shape[0]:>12,}')
print(f'{"Số cột":<35} {df_raw.shape[1]:>12} {df.shape[1]:>12}')
print(f'{"Số feature N*":<35} {len([c for c in df_raw.columns if c.startswith("N")]):>12} {len(feature_cols):>12}')
print(f'{"Missing trong numeric features":<35} {df_raw[[c for c in df_raw.columns if c.startswith("N")] + ["SIZE"]].isnull().sum().sum():>12,} {df[numeric_feats].isnull().sum().sum():>12,}')
print(f'{"Missing trong SECTOR":<35} {df_raw["SECTOR"].isnull().sum():>12} {df["SECTOR"].isnull().sum():>12}')

---
# PHẦN 3: EDA – PHÂN TÍCH KHÁM PHÁ DỮ LIỆU

## Phân phối nhãn Distress (Class Distribution)

In [ ]:
counts = df['distress_flag'].value_counts()
labels = ['Không KQTC (0)', 'KQTC (1)']
colors = ['#2ECC71', '#E74C3C']

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Biểu đồ cột
bars = axes[0].bar(labels, counts.values, color=colors, alpha=0.85, edgecolor='white', width=0.5)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{val:,}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Số quan sát')
axes[0].set_title('Phân phối lớp nhãn', fontsize=12, fontweight='bold')
axes[0].set_ylim(0, counts.max() * 1.15)

# Biểu đồ tròn
wedges, texts, autotexts = axes[1].pie(
    counts.values, labels=labels, colors=colors,
    autopct='%1.1f%%', startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontsize(12)
    at.set_fontweight('bold')
axes[1].set_title('Tỉ lệ phần trăm', fontsize=12, fontweight='bold')

plt.suptitle('Phân phối Nhãn (distress_flag)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

ratio = counts[1] / counts[0] * 100
print(f'Tỉ lệ mất cân bằng: {counts[0]:,} không KQTC vs {counts[1]:,} KQTC (Số quan sát KQTC bằng {ratio:.1f}% số quan sát không KQTC)')

Dữ liệu mất cân bằng đáng kể – chỉ khoảng ~30% quan sát thuộc lớp KQTC. Điều này đặt ra yêu cầu phải xử lý imbalance (SMOTE) trước khi huấn luyện mô hình

## Xu hướng KQTC theo năm

In [ ]:
# Tỉ lệ KQTC theo từng năm
yearly = df.groupby('yearReport')['distress_flag'].agg(['sum', 'count'])
yearly['rate'] = yearly['sum'] / yearly['count'] * 100

fig, ax1 = plt.subplots(figsize=(12, 5))

# Cột số lượng
ax1.bar(yearly.index, yearly['count'], color='#AED6F1', alpha=0.7, label='Tổng quan sát', zorder=1)
ax1.bar(yearly.index, yearly['sum'], color='#E74C3C', alpha=0.85, label='Số KQTC', zorder=2)
ax1.set_xlabel('Năm', fontsize=11)
ax1.set_ylabel('Số quan sát', fontsize=11)
ax1.legend(loc='upper left', fontsize=9)

# Đường tỉ lệ %
ax2 = ax1.twinx()
ax2.plot(yearly.index, yearly['rate'], color='#2C3E50', marker='o',
         linewidth=2, markersize=5, label='Tỉ lệ KQTC (%)', zorder=3)
ax2.set_ylabel('Tỉ lệ KQTC (%)', fontsize=11)
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax2.legend(loc='upper right', fontsize=9)

# Đánh dấu giai đoạn COVID
ax1.axvspan(2020, 2022, alpha=0.08, color='orange', label='COVID-19')
ax1.text(2020.5, ax1.get_ylim()[1]*0.9, 'COVID-19', fontsize=9, color='darkorange')

plt.title('Xu hướng Kiệt Quệ Tài Chính theo Năm', fontsize=13, fontweight='bold')
plt.xticks(yearly.index, rotation=45)
plt.tight_layout()
plt.savefig('fig_trend_by_year.png', dpi=150, bbox_inches='tight')
plt.show()

## Tỷ lệ KQTC theo ngành (SECTOR)

In [ ]:
# Tỷ lệ KQTC theo từng ngành
sec_stats = (df.groupby('SECTOR')['distress_flag']
               .agg(['sum', 'count', 'mean'])
               .rename(columns={'sum':'so_KQTC','count':'tong_qs','mean':'ty_le'}))
sec_stats['ty_le_pct'] = sec_stats['ty_le'] * 100
sec_stats = sec_stats.sort_values('ty_le_pct', ascending=True)

fig, ax = plt.subplots(figsize=(11, 5.5))
colors_bar = ['#E74C3C' if r > 25 else '#F39C12' if r > 20 else '#27AE60'
              for r in sec_stats['ty_le_pct']]
bars = ax.barh(sec_stats.index, sec_stats['ty_le_pct'], color=colors_bar,
               alpha=0.85, edgecolor='white')

# Đường tỉ lệ trung bình toàn bộ
overall = df['distress_flag'].mean() * 100
ax.axvline(overall, color='#2C3E50', linestyle='--', linewidth=1.5,
           label=f'Tỉ lệ trung bình toàn bộ: {overall:.1f}%')

for bar, (idx, row) in zip(bars, sec_stats.iterrows()):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f' {row["ty_le_pct"]:.1f}%',
            va='center', fontsize=10)

ax.set_xlabel('Tỉ lệ KQTC (%)', fontsize=11)
ax.set_title('Tỉ lệ Kiệt Quệ Tài Chính theo Ngành',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim(0, sec_stats['ty_le_pct'].max() * 1.20)
plt.tight_layout()
plt.savefig('fig_kqtc_by_sector.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nBảng tỉ lệ KQTC theo ngành:')
print(sec_stats[['so_KQTC','tong_qs','ty_le_pct']].round(2).to_string())
print(f'\nNgành rủi ro cao nhất: {sec_stats["ty_le_pct"].idxmax()} '
      f'({sec_stats["ty_le_pct"].max():.1f}%)')
print(f'Ngành rủi ro thấp nhất: {sec_stats["ty_le_pct"].idxmin()} '
      f'({sec_stats["ty_le_pct"].min():.1f}%)')
print(f'Chênh lệch: {sec_stats["ty_le_pct"].max()/sec_stats["ty_le_pct"].min():.2f}x')

## Phân phối SIZE giữa 2 nhóm

In [ ]:
size_0 = df.loc[df['distress_flag']==0, 'SIZE']
size_1 = df.loc[df['distress_flag']==1, 'SIZE']

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# (1) Histogram
axes[0].hist(size_0, bins=50, alpha=0.55, density=True,
             color='#2ECC71', label=f'Không KQTC (median = {size_0.median():.2f})')
axes[0].hist(size_1, bins=50, alpha=0.55, density=True,
             color='#E74C3C', label=f'KQTC (median = {size_1.median():.2f})')
axes[0].axvline(size_0.median(), color='#27AE60', linestyle='--', linewidth=1.5)
axes[0].axvline(size_1.median(), color='#C0392B', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('SIZE = ln(Total Assets / CPI)', fontsize=11)
axes[0].set_ylabel('Mật độ', fontsize=11)
axes[0].set_title('Phân phối SIZE', fontsize=11, fontweight='bold')
axes[0].legend(fontsize=9)

# (2) Boxplot
df_plot = df.copy()
df_plot['Nhóm'] = df_plot['distress_flag'].map({0:'Không KQTC', 1:'KQTC'})
sns.boxplot(data=df_plot, x='Nhóm', y='SIZE',
            palette={'Không KQTC':'#2ECC71','KQTC':'#E74C3C'},
            width=0.45, fliersize=2, ax=axes[1])
axes[1].set_title('Boxplot SIZE theo nhóm', fontsize=11, fontweight='bold')
axes[1].set_xlabel('')

plt.suptitle('Quy mô doanh nghiệp (SIZE) và Kiệt Quệ Tài Chính',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig_size_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nThống kê SIZE:')
print(f'  - Nhóm Không KQTC: median = {size_0.median():.3f}, mean = {size_0.mean():.3f}')
print(f'  - Nhóm KQTC:       median = {size_1.median():.3f}, mean = {size_1.mean():.3f}')
print(f'  - Chênh lệch median = {size_0.median()-size_1.median():.3f} '
      f'(~{np.exp(size_0.median()-size_1.median()):.2f}x về Total Assets)')

## Trực quan hóa phân phối các biến chính (FD vs Non-FD)

In [ ]:
# lấy 1 vài biến quan trọng làm mẫu phân phối
key_vars = [
    'N1_EBITTA', 'N4_RETA', 'N10_CACL', 'N11_WCTA',
    'N24_TLTA', 'N26_TLSE', 'N28_GTA', 'N32_GTS'
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, var in enumerate(key_vars):
    # Clip ở 1%-99% CHỈ để vẽ (không sửa df gốc)
    lb = df[var].quantile(0.01)
    ub = df[var].quantile(0.99)

    for label, color, alpha in [(0, '#2ECC71', 0.6), (1, '#E74C3C', 0.6)]:
        data = df[df['distress_flag'] == label][var].dropna().clip(lb, ub)
        axes[i].hist(data, bins=40, alpha=alpha, color=color, density=True,
                     label='Không KQTC' if label == 0 else 'KQTC')
    axes[i].set_title(var, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Giá trị')
    axes[i].set_ylabel('Mật độ' if i % 4 == 0 else '')
    if i == 0:
        axes[i].legend(fontsize=8)

plt.suptitle('Phân phối các chỉ tiêu tài chính: Nhóm KQTC vs Không KQTC\n'
             '(đã clip [1%, 99%] để dễ quan sát)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_distribution_fd_nonfd.png', dpi=150, bbox_inches='tight')
plt.show()

## Boxplot so sánh nhóm FD vs Non-FD

In [ ]:
# Boxplot cho một số biến đặc trưng cho từng nhóm chỉ tiêu
box_vars = ['N1_EBITTA', 'N4_RETA', 'N10_CACL', 'N11_WCTA', 'N24_TLTA', 'N28_GTA']

# Dung df_plot rieng de khong lam ban df goc
df_plot = df.copy()
df_plot['Nhom'] = df_plot['distress_flag'].map({0: 'Khong KQTC', 1: 'KQTC'})

# Clip các biến ở [1%, 99%] để không bị outliers kéo giãn
for var in box_vars:
    lb = df_plot[var].quantile(0.01)
    ub = df_plot[var].quantile(0.99)
    df_plot[var] = df_plot[var].clip(lb, ub)

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
axes = axes.flatten()
palette = {'Khong KQTC': '#2ECC71', 'KQTC': '#E74C3C'}

for i, var in enumerate(box_vars):
    sns.boxplot(data=df_plot, x='Nhom', y=var, palette=palette,
                width=0.5, fliersize=2, ax=axes[i])
    axes[i].set_title(var, fontsize=11)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Gia tri')

plt.suptitle('So Sanh Phan Phoi: KQTC vs Khong KQTC (đã clip [1%, 99%] để dễ quan sát)',
             fontsize=13)
plt.tight_layout()
plt.savefig('fig_boxplot_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Heatmap tương quan

In [ ]:
# Thêm SIZE vào danh sách biến tương quan
corr_vars = [
    'N1_EBITTA', 'N2_NITA', 'N4_RETA', 'N7_EBITTS',
    'N10_CACL', 'N11_WCTA', 'N12_CLTA', 'N24_TLTA',
    'N26_TLSE', 'N28_GTA', 'N32_GTS',
    'SIZE',
    'distress_flag'
]

corr_matrix = df[corr_vars].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True, fmt='.2f', annot_kws={'size': 8},
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5,
    cbar_kws={'shrink': 0.8, 'label': 'Hệ số tương quan'},
    ax=ax
)

ax.set_title('Heatmap Tương Quan Giữa Các Biến Tài Chính',
             fontsize=13, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('fig_heatmap_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# Tính tương quan với distress_flag
corr_with_target = corr_matrix['distress_flag'].drop('distress_flag')
# Lấy top 10 theo độ lớn (abs)
top10 = corr_with_target.reindex(corr_with_target.abs().sort_values(ascending=False).index).head(10)

# Trực quan hóa: bar chart với màu phân biệt sign
fig, ax = plt.subplots(figsize=(9, 5))
top10_sorted = top10.sort_values()  # sort tăng dần để bar dài nhất ở dưới cùng
colors_bar = ['#27AE60' if v < 0 else '#E74C3C' for v in top10_sorted]
bars = ax.barh(top10_sorted.index, top10_sorted.values,
               color=colors_bar, alpha=0.85, edgecolor='white')

# Label giá trị
for bar, val in zip(bars, top10_sorted.values):
    x_pos = bar.get_width()
    offset = 0.005 if val >= 0 else -0.005
    ha = 'left' if val >= 0 else 'right'
    ax.text(x_pos + offset, bar.get_y() + bar.get_height()/2,
            f'{val:+.3f}', va='center', ha=ha, fontsize=9)

ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Hệ số tương quan với distress_flag', fontsize=11)
ax.set_title('Top 10 biến tương quan mạnh nhất với distress_flag\n'
             '(xanh = tương quan âm, đỏ = tương quan dương)',
             fontsize=12, fontweight='bold')
ax.set_xlim(top10_sorted.min() * 1.25, top10_sorted.max() * 1.25)
plt.tight_layout()
plt.savefig('fig_top10_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# In bảng kèm theo
print('\nTop 10 biến tương quan với distress_flag (sắp xếp theo |corr|):')
print(top10.to_frame('correlation').round(4).to_string())

---
# PHẦN 4: XÂY DỰNG MÔ HÌNH

In [ ]:

import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')

# Sklearn – Base
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Sklearn – Preprocessing
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder

# Sklearn – Models
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

# Sklearn – Tuning & CV
from sklearn.model_selection import (
    StratifiedKFold, RandomizedSearchCV, cross_val_predict
)

# Sklearn – Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, roc_curve, precision_recall_curve
)

# Imbalanced-learn
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Scipy – Distributions for RandomizedSearch
from scipy.stats import loguniform, randint

# Thanh progress bar
from tqdm.auto import tqdm
import joblib
from contextlib import contextmanager

print("Import thư viện thành công.")

Import thư viện thành công.


## Tạo target KQTC

In [ ]:
# ============================================================
# Tạo target: KQTC trong 4 quý tới (OR logic)
# ============================================================
# Target = 1 nếu công ty có ÍT NHẤT 1 quý KQTC trong t+1..t+4
# Target = 0 nếu cả 4 quý đều KHÔNG KQTC
#
# Tại sao OR thay vì shift(-1)?
#   - Câu hỏi thực tế: "Năm tới công ty có gặp rủi ro không?"
#   - 1 quý phục hồi tạm thời không "ăn gian" label
#   - Tỉ lệ positive tăng từ ~24% → ~45% (gần cân bằng hơn)
#   - Persistence giảm: model buộc phải học pattern thực sự

print("=" * 60)
print("Tạo target KQTC trong 4 quý tới")
print("=" * 60)

# df đã được load và tiền xử lý từ Phần 1–3 (có distress_flag, symbol, Date, yearReport)
df = df.sort_values(['symbol', 'Date']).reset_index(drop=True)

# OR của 4 quý tới: lấy shift(-1) đến shift(-4), sau đó max theo hàng
future_flags = pd.concat(
    [df.groupby('symbol')['distress_flag'].shift(-k) for k in [1, 2, 3, 4]],
    axis=1
)
future_flags.columns = ['flag_t1', 'flag_t2', 'flag_t3', 'flag_t4']
df['target'] = future_flags.max(axis=1)

# Bỏ 4 dòng cuối mỗi công ty (không đủ 4 quý tới để tính target)
df = df.dropna(subset=['target']).reset_index(drop=True)
df['target'] = df['target'].astype(int)

# Kiểm tra persistence (tương quan với flag hiện tại)
persistence = df['distress_flag'].corr(df['target'])
print(f"Số mẫu sau khi tạo target : {len(df):,}")
print(f"Tỉ lệ KQTC trong 4 quý tới: {df['target'].mean()*100:.1f}%")
print(f"Tỉ lệ KQTC hiện tại (t)   : {df['distress_flag'].mean()*100:.1f}%")
print(f"Persistence (corr flag_t vs target): {persistence:.3f}")


Tạo target KQTC trong 4 quý tới
Số mẫu sau khi tạo target : 16,352
Tỉ lệ KQTC trong 4 quý tới: 43.7%
Tỉ lệ KQTC hiện tại (t)   : 22.9%
Persistence (corr flag_t vs target): 0.442


## Time-based Split

In [ ]:
# ============================================================
# Time-based Split (Train ≤ 2022, Test ≥ 2023)
# ============================================================
# Mô phỏng kịch bản dự báo thực tế: huấn luyện trên quá khứ,
# đánh giá trên tương lai. Tránh data leakage giữa các quý của
# cùng công ty (random split có thể cho quý t+2 vào train, quý t vào test).

print("\n" + "=" * 60)
print("Time-based Split")
print("=" * 60)

# Định nghĩa feature columns (đồng bộ với kết quả drop N16 ở Phần 2)
feature_cols      = [c for c in df.columns if c.startswith('N')]
numeric_features  = feature_cols + ['SIZE']
categorical_features = ['SECTOR']
all_features      = numeric_features + categorical_features

mask_train = df['yearReport'] <= 2022
mask_test  = df['yearReport'] >= 2023

X_train = df.loc[mask_train, all_features].reset_index(drop=True)
y_train = df.loc[mask_train, 'target'].values

X_test  = df.loc[mask_test,  all_features].reset_index(drop=True)
y_test  = df.loc[mask_test,  'target'].values

print(f"Train : {len(X_train):,} mẫu | KQTC = {y_train.mean()*100:.1f}%"
      f" | yearReport ≤ 2022")
print(f"Test  : {len(X_test):,} mẫu | KQTC = {y_test.mean()*100:.1f}%"
      f" | yearReport ≥ 2023")
print(f"\nFeatures: {len(numeric_features)} numeric + {len(categorical_features)} categorical"
      f" = {len(all_features)} tổng")

# Phân phối nhãn theo năm (để xác nhận split hợp lý)
yearly_check = df.groupby('yearReport')['target'].agg(['count', 'mean'])
yearly_check.columns = ['n_obs', 'kqtc_rate']
yearly_check['kqtc_rate'] = (yearly_check['kqtc_rate'] * 100).round(1)
yearly_check['split'] = np.where(yearly_check.index <= 2022, 'TRAIN', 'TEST')
print(f"\nPhân phối theo năm:\n{yearly_check.to_string()}")


## Định nghĩa Preprocessor

In [ ]:

# ============================================================
# Định nghĩa Preprocessor CHUNG
# ============================================================
# Preprocessor này được dùng chung cho CẢ 6 MODEL.
# Pipeline xử lý:
#   Numeric  : Winsorize[1%,99%] → Impute(median) → RobustScaler
#   Categoric: OneHotEncoder(drop='first', handle_unknown='ignore')
#
# Tất cả đều FIT CHỈ TRÊN TRAIN (sklearn Pipeline đảm bảo điều này).

print("\n" + "=" * 60)
print("Định nghĩa Preprocessor chung")
print("=" * 60)


class Winsorizer(BaseEstimator, TransformerMixin):
    """
    Clip giá trị cực đoan về [lower_quantile, upper_quantile] của tập train.
    Outliers chứa tín hiệu KQTC quan trọng → không loại bỏ, chỉ giới hạn.
    Fit: tính phân vị trên X_train. Transform: clip cả train lẫn test.
    """
    def __init__(self, lower: float = 0.01, upper: float = 0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.lb_ = np.nanpercentile(X, self.lower * 100, axis=0)
        self.ub_ = np.nanpercentile(X, self.upper * 100, axis=0)
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lb_, self.ub_)

    def get_feature_names_out(self, input_features=None):
        return input_features if input_features is not None else np.array([])


# Pipeline xử lý biến số
numeric_pipe = Pipeline([
    ('winsorize', Winsorizer(0.01, 0.99)),          # Clip outlier, giữ tín hiệu KQTC
    ('imputer',   SimpleImputer(strategy='median')), # Xử lý NaN còn lại sau fillna theo symbol
    ('scaler',    RobustScaler()),                   # Chuẩn hóa robust với heavy-tail
])

# Preprocessor chung cho cả 6 model (ColumnTransformer)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipe, numeric_features),
        ('cat', OneHotEncoder(
            drop='first',            # Tránh multicollinearity (dummy trap)
            sparse_output=False,     # Trả về dense array
            handle_unknown='ignore'  # Test có category lạ → không lỗi
        ), categorical_features),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)

# Kiểm tra preprocessor hoạt động đúng trước khi dùng cho model
_X_check = preprocessor.fit_transform(X_train)
print(f" Preprocessor OK. Output shape: {_X_check.shape}")
print(f"   Numeric features  : {len(numeric_features)}")
print(f"   Categorical (OHE) : {preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features).tolist()}")
print(f"   Total output cols : {_X_check.shape[1]}")
del _X_check  # Dùng xong, xóa để tránh nhầm lẫn (preprocessor sẽ fit lại trong Pipeline)

# Reset preprocessor về chưa fit, để mỗi ImbPipeline tự fit trên train của nó
from sklearn.base import clone
preprocessor = clone(preprocessor)

print("\n preprocessor, X_train, X_test, y_train, y_test đã sẵn sàng.")
print("   Các thành viên dùng chung các biến này cho Bước 4–6.")


# ============================================================
# HÀM TIỆN ÍCH DÙNG CHUNG (các model dùng lại)
# ============================================================

@contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback

    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
        tqdm_object.close()


def find_best_threshold_f1(y_true: np.ndarray, y_score: np.ndarray):
    """
    Tìm threshold tối ưu hóa F1 từ OOF probabilities trên tập TRAIN.
    Không bao giờ dùng y_test ở bước này.
    """
    prec, rec, thresholds = precision_recall_curve(y_true, y_score)
    if len(thresholds) == 0:
        return 0.5, 0.0
    f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
    best_idx = int(np.nanargmax(f1s))
    return float(thresholds[best_idx]), float(f1s[best_idx])


def evaluate_on_test(estimator, X_test, y_test, threshold: float,
                     model_name: str = "Model"):
    """
    Bước 6: Đánh giá model trên test set với threshold đã tune.
    In 6 metrics + ROC curve + Confusion Matrix.
    Trả về dict kết quả để gộp sau.
    """
    y_proba = estimator.predict_proba(X_test)[:, 1]
    y_pred  = (y_proba >= threshold).astype(int)

    result = {
        'model'    : model_name,
        'threshold': round(threshold, 4),
        'accuracy' : round(accuracy_score(y_test, y_pred), 4),
        'precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'recall'   : round(recall_score(y_test, y_pred, zero_division=0), 4),
        'f1'       : round(f1_score(y_test, y_pred, zero_division=0), 4),
        'roc_auc'  : round(roc_auc_score(y_test, y_proba), 4),
        'pr_auc'   : round(average_precision_score(y_test, y_proba), 4),
    }

    # In kết quả
    print(f"\n{'='*62}")
    print(f"  {model_name}  |  threshold = {threshold:.4f}")
    print(f"{'='*62}")
    for k, v in result.items():
        if k not in ('model', 'threshold'):
            print(f"  {k:12s}: {v:.4f}")

    print("\n  Classification Report:")
    print(classification_report(y_test, y_pred,
                                 target_names=['Không KQTC', 'KQTC'],
                                 zero_division=0))

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].plot(fpr, tpr, color='#2980B9', lw=2,
                 label=f'ROC AUC = {result["roc_auc"]:.3f}')
    axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, lw=1)
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title(f'ROC Curve – {model_name}')
    axes[0].legend(loc='lower right')

    # Confusion Matrix
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, y_pred),
        display_labels=['Không KQTC', 'KQTC']
    ).plot(ax=axes[1], colorbar=False, cmap='Blues')
    axes[1].set_title(f'Confusion Matrix – {model_name}\n(threshold = {threshold:.4f})')

    plt.suptitle(model_name, fontweight='bold')
    plt.tight_layout()
    plt.show()

    return result




## 4.1 Logistic Regression

In [ ]:
# ============================================================
#   Model 1: LOGISTIC REGRESSION
# ============================================================

print("\n" + "=" * 62)
print("  MODEL 1 – LOGISTIC REGRESSION")
print("=" * 62)

# -------------------------------------------------------
# Build ImbPipeline + RandomizedSearchCV
# -------------------------------------------------------
# Thiết kế search space tối ưu cho LR trên dữ liệu tài chính:
#
#  Combo 1: L2 + lbfgs  → baseline mạnh, ổn định, nhanh
#  Combo 2: L1 + saga   → feature selection ngầm, loại biến không liên quan
#  Combo 3: ElasticNet + saga → kết hợp L1+L2, linh hoạt nhất
#
#  class_weight='balanced': tự động xử lý imbalance qua cost-sensitive learning
#  SMOTE với k_neighbors khác nhau: tăng robustness của oversampling
#
#  Kỹ thuật nâng cao:
#  - Thêm SMOTE__k_neighbors vào search để tìm mức oversample tốt nhất
#  - Kết hợp class_weight với SMOTE để có 2 chiến lược xử lý imbalance

lg_pipe = ImbPipeline(steps=[
    ('preprocessor', preprocessor),                          # ← từ Bước 3
    ('smote', SMOTE(random_state=42)),
    ('clf', LogisticRegression(max_iter=2000, random_state=42)),
])

lg_param_dist = [
    # Combo 1: L2 + lbfgs (nhanh, tốt với dữ liệu dense)
    {
        'smote__k_neighbors' : [3, 5, 7],
        'clf__C'             : loguniform(1e-3, 50),
        'clf__penalty'       : ['l2'],
        'clf__solver'        : ['lbfgs'],
        'clf__class_weight'  : [None, 'balanced'],
    },
    # Combo 2: L1 + saga (sparse solution, feature selection ngầm)
    {
        'smote__k_neighbors' : [3, 5, 7],
        'clf__C'             : loguniform(1e-3, 50),
        'clf__penalty'       : ['l1'],
        'clf__solver'        : ['saga'],
        'clf__class_weight'  : [None, 'balanced'],
    },
    # Combo 3: ElasticNet + saga (kết hợp L1+L2, linh hoạt)
    {
        'smote__k_neighbors' : [3, 5, 7],
        'clf__C'             : loguniform(1e-3, 50),
        'clf__penalty'       : ['elasticnet'],
        'clf__solver'        : ['saga'],
        'clf__l1_ratio'      : [0.1, 0.3, 0.5, 0.7, 0.9],
        'clf__class_weight'  : [None, 'balanced'],
    },
]

cv_5fold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring  = {
    'f1'       : 'f1',
    'roc_auc'  : 'roc_auc',
    'precision': 'precision',
    'recall'   : 'recall',
    'accuracy' : 'accuracy',
}


lg_search = RandomizedSearchCV(
    estimator          = lg_pipe,
    param_distributions= lg_param_dist,
    n_iter             = 30,          # 30 iter × 5 fold = 150 fits – đủ khám phá
    scoring            = scoring,
    refit              = 'f1',        # Chọn best model theo F1
    cv                 = cv_5fold,
    n_jobs             = -1,
    verbose            = 0,
    random_state       = 42,
    return_train_score = False,
    error_score        = 'raise',
)

print(">>> Tuning Logistic Regression...")
t0 = time.time()

total_fit_lg = lg_search.n_iter * cv_5fold.get_n_splits()
with tqdm_joblib(tqdm(desc="LG / Logistic Regression", total=total_fit_lg)):
    lg_search.fit(X_train, y_train)

elapsed_lg = round(time.time() - t0, 2)

print(f"\n Thời gian tuning: {elapsed_lg}s")
print(f"Best params: {lg_search.best_params_}")
print(f"Best CV F1 : {lg_search.best_score_:.4f}")

# CV scores của best params
cv_res_lg  = pd.DataFrame(lg_search.cv_results_)
best_idx_lg = lg_search.best_index_
print("\nCV Mean Scores (best params):")
for m in ['f1', 'roc_auc', 'precision', 'recall', 'accuracy']:
    print(f"  {m:12s}: {cv_res_lg.loc[best_idx_lg, f'mean_test_{m}']:.4f}")

# -------------------------------------------------------
# Threshold Tuning (OOF trên train)
# -------------------------------------------------------
# Tính OOF probability trên TRAIN (không đụng test)
# Sau đó tìm threshold tối đa hóa F1 → dùng cho bước 6

print("\n>>> Threshold tuning – Logistic Regression (OOF trên train)...")
oof_proba_lg = cross_val_predict(
    lg_search.best_estimator_,
    X_train, y_train,
    cv=cv_5fold, method='predict_proba', n_jobs=-1
)[:, 1]

lg_best_thr, lg_oof_f1 = find_best_threshold_f1(y_train, oof_proba_lg)
print(f"Best OOF threshold (F1-max): {lg_best_thr:.4f}")
print(f"OOF Train F1 tại threshold : {lg_oof_f1:.4f}")

# Vẽ F1 theo threshold để kiểm tra
prec_lg, rec_lg, thr_lg = precision_recall_curve(y_train, oof_proba_lg)
f1s_lg = 2 * prec_lg[:-1] * rec_lg[:-1] / (prec_lg[:-1] + rec_lg[:-1] + 1e-12)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(thr_lg, f1s_lg, color='#2980B9', lw=1.5, label='F1 (OOF train)')
ax.plot(thr_lg, prec_lg[:-1], color='#27AE60', lw=1, linestyle='--', label='Precision')
ax.plot(thr_lg, rec_lg[:-1], color='#E74C3C', lw=1, linestyle='--', label='Recall')
ax.axvline(lg_best_thr, color='black', linestyle=':', lw=1.5,
           label=f'Best thr = {lg_best_thr:.3f}')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('Threshold Tuning – Logistic Regression (OOF Train)')
ax.legend(); ax.set_xlim(0, 1); plt.tight_layout(); plt.show()

# -------------------------------------------------------
# Đánh giá Test
# -------------------------------------------------------

result_lg = evaluate_on_test(
    estimator  = lg_search.best_estimator_,
    X_test     = X_test,
    y_test     = y_test,
    threshold  = lg_best_thr,
    model_name = 'Logistic Regression',
)
result_lg['train_time_s'] = elapsed_lg
result_lg['oof_f1']       = round(lg_oof_f1, 4)


## 4.2 ANN

In [ ]:
# ============================================================
#   Model 2: ANN (MLPClassifier)
# ============================================================

print("\n" + "=" * 62)
print("  MODEL 2 – ANN (MLPClassifier)")
print("=" * 62)

# -------------------------------------------------------
# Build ImbPipeline + RandomizedSearchCV
# -------------------------------------------------------
# Thiết kế search space tối ưu cho ANN trên tabular financial data:
#
#  Kiến trúc: từ 1 đến 3 lớp ẩn, size 32–256
#    - Nông (1 lớp): generalize tốt hơn khi dữ liệu không quá lớn
#    - Sâu (2-3 lớp): học được interaction phức tạp hơn
#
#  Regularization:
#    - alpha (L2): kiểm soát overfitting, quan trọng với tabular data
#    - early_stopping: tránh overfit theo epoch
#
#  Learning dynamics:
#    - learning_rate_init: ảnh hưởng lớn đến hội tụ
#    - batch_size: nhỏ hơn → noisy gradient → regularization ngầm
#    - activation: relu (sparse), tanh (smoother gradients)
#
#  SMOTE kết hợp class_weight='balanced' không áp dụng được với MLP
#  → dùng SMOTE trong pipeline để xử lý imbalance

ann_pipe = ImbPipeline(steps=[
    ('preprocessor', preprocessor),     # ← từ Bước 3 (quan trọng: ANN cần scale)
    ('smote', SMOTE(random_state=42)),
    ('clf', MLPClassifier(
        solver             = 'adam',
        early_stopping     = True,       # Dừng sớm khi val loss không cải thiện
        validation_fraction= 0.1,        # 10% train làm validation cho early stopping
        n_iter_no_change   = 20,         # Kiên nhẫn hơn để tìm minimum thực
        max_iter           = 500,        # Đủ epoch để hội tụ
        random_state       = 42,
    )),
])

ann_param_dist = {
    # SMOTE
    'smote__k_neighbors': [3, 5, 7],

    # Kiến trúc mạng
    'clf__hidden_layer_sizes': [
        (64,),              # 1 lớp nhỏ – baseline nhanh
        (128,),             # 1 lớp trung bình
        (256,),             # 1 lớp lớn
        (128, 64),          # 2 lớp – phổ biến nhất cho tabular
        (256, 128),         # 2 lớp lớn
        (64, 32),           # 2 lớp nhỏ – tránh overfit
        (128, 64, 32),      # 3 lớp – phức tạp hơn
        (256, 128, 64),     # 3 lớp lớn – cần nhiều data
        (64, 64),           # 2 lớp đều nhau
        (128, 128),         # 2 lớp đều nhau, lớn hơn
    ],

    # Activation function
    'clf__activation': ['relu', 'tanh'],

    # Regularization L2 (weight decay)
    'clf__alpha': loguniform(1e-5, 1e-1),

    # Learning rate
    'clf__learning_rate_init': loguniform(5e-5, 5e-3),

    # Batch size – ảnh hưởng đến noise và tốc độ
    'clf__batch_size': [32, 64, 128, 256],
}

ann_search = RandomizedSearchCV(
    estimator          = ann_pipe,
    param_distributions= ann_param_dist,
    n_iter             = 30,          # ANN chậm hơn LR → 30 iter là hợp lý
    scoring            = scoring,
    refit              = 'f1',
    cv                 = cv_5fold,
    n_jobs             = -1,          # Parallel nếu có nhiều CPU; đặt 1 nếu OOM
    verbose            = 0,
    random_state       = 42,
    return_train_score = False,
    error_score        = 'raise',
)

print(">>> Tuning ANN (MLPClassifier)...")
t0 = time.time()

total_fit_ann = ann_search.n_iter * cv_5fold.get_n_splits()
with tqdm_joblib(tqdm(desc="ANN / MLPClassifier", total=total_fit_ann)):
    ann_search.fit(X_train, y_train)

elapsed_ann = round(time.time() - t0, 2)

print(f"\n Thời gian tuning: {elapsed_ann}s")
print(f"Best params: {ann_search.best_params_}")
print(f"Best CV F1 : {ann_search.best_score_:.4f}")

cv_res_ann   = pd.DataFrame(ann_search.cv_results_)
best_idx_ann = ann_search.best_index_
print("\nCV Mean Scores (best params):")
for m in ['f1', 'roc_auc', 'precision', 'recall', 'accuracy']:
    print(f"  {m:12s}: {cv_res_ann.loc[best_idx_ann, f'mean_test_{m}']:.4f}")

# -------------------------------------------------------
# Threshold Tuning (OOF trên train)
# -------------------------------------------------------

print("\n>>> Threshold tuning – ANN (OOF trên train)...")
oof_proba_ann = cross_val_predict(
    ann_search.best_estimator_,
    X_train, y_train,
    cv=cv_5fold, method='predict_proba', n_jobs=-1
)[:, 1]

ann_best_thr, ann_oof_f1 = find_best_threshold_f1(y_train, oof_proba_ann)
print(f"Best OOF threshold (F1-max): {ann_best_thr:.4f}")
print(f"OOF Train F1 tại threshold : {ann_oof_f1:.4f}")

# Vẽ F1 theo threshold
prec_ann, rec_ann, thr_ann = precision_recall_curve(y_train, oof_proba_ann)
f1s_ann = 2 * prec_ann[:-1] * rec_ann[:-1] / (prec_ann[:-1] + rec_ann[:-1] + 1e-12)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(thr_ann, f1s_ann, color='#8E44AD', lw=1.5, label='F1 (OOF train)')
ax.plot(thr_ann, prec_ann[:-1], color='#27AE60', lw=1, linestyle='--', label='Precision')
ax.plot(thr_ann, rec_ann[:-1], color='#E74C3C', lw=1, linestyle='--', label='Recall')
ax.axvline(ann_best_thr, color='black', linestyle=':', lw=1.5,
           label=f'Best thr = {ann_best_thr:.3f}')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('Threshold Tuning – ANN (OOF Train)')
ax.legend(); ax.set_xlim(0, 1); plt.tight_layout(); plt.show()

# -------------------------------------------------------
# Đánh giá Test
# -------------------------------------------------------

result_ann = evaluate_on_test(
    estimator  = ann_search.best_estimator_,
    X_test     = X_test,
    y_test     = y_test,
    threshold  = ann_best_thr,
    model_name = 'ANN (MLPClassifier)',
)
result_ann['train_time_s'] = elapsed_ann
result_ann['oof_f1']       = round(ann_oof_f1, 4)




In [ ]:
# ============================================================
# TỔNG HỢP KẾT QUẢ 2 MODEL (LR + ANN)
# ============================================================
# Lưu vào biến result_lg và result_ann để nhóm trưởng gộp sau

print("\n" + "=" * 62)
print("  KẾT QUẢ TỔNG HỢP: LR + ANN")
print("=" * 62)

summary_cols = ['model', 'threshold', 'accuracy', 'precision',
                'recall', 'f1', 'roc_auc', 'pr_auc', 'oof_f1', 'train_time_s']

# Ví dụ về cách thêm các kết quả mô hình khác (giả định có result_dt, result_rf, etc.)
# Bạn sẽ thay thế 'None' bằng dict kết quả thực tế của các mô hình đó.
summary_df = pd.DataFrame([
    result_lg,
    result_ann,
    # result_dt, # Thêm kết quả của Decision Tree vào đây
    # result_rf, # Thêm kết quả của Random Forest vào đây
    # result_svm, # Thêm kết quả của SVM vào đây
    # result_xgb, # Thêm kết quả của XGBoost vào đây
])[summary_cols]

summary_df = summary_df.sort_values('f1', ascending=False).reset_index(drop=True)
print(summary_df.to_string(index=False))
summary_df


## 4.3 Decision Tree

In [ ]:
# ============================================================
#   Model 3: DECISION TREE
# ============================================================
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, loguniform
from sklearn.base import clone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

print("\n" + "=" * 62)
print("  MODEL 3 – DECISION TREE")
print("=" * 62)

In [ ]:
# -------------------------------------------------------
# Time-based CV trong train
# -------------------------------------------------------
# Dùng expanding window theo năm:
# train <= 2016 -> val = 2017
# train <= 2017 -> val = 2018
# ...
# train <= 2021 -> val = 2022

train_year = df.loc[mask_train, 'yearReport'].reset_index(drop=True)

valid_years = [2017, 2018, 2019, 2020, 2021, 2022]
cv_time = []

for val_year in valid_years:
    tr_idx = np.flatnonzero(train_year < val_year)
    va_idx = np.flatnonzero(train_year == val_year)
    if len(tr_idx) > 0 and len(va_idx) > 0:
        cv_time.append((tr_idx, va_idx))

print("Time folds:")
for i, (tr_idx, va_idx) in enumerate(cv_time, 1):
    print(f"  Fold {i}: train={len(tr_idx):,} | val={len(va_idx):,} | val_year={train_year.iloc[va_idx].iloc[0]}")


In [ ]:
# -------------------------------------------------------
# Pipeline + RandomizedSearchCV
# -------------------------------------------------------
# Giữ nguyên preprocessor đã có, chỉ thay phần model.
# Không dùng SMOTE cho DT.

dt_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', DecisionTreeClassifier(random_state=42)),
])

dt_param_dist = {
    'clf__criterion': ['gini', 'entropy', 'log_loss'],
    'clf__max_depth': [3, 4, 5, 6, 8, 10, 12, None],
    'clf__min_samples_split': randint(40, 401),
    'clf__min_samples_leaf': randint(20, 201),
    'clf__max_leaf_nodes': [8, 12, 16, 24, 32, 48, 64, None],
    'clf__max_features': [None, 'sqrt', 'log2', 0.5, 0.7, 0.9],
    'clf__class_weight': [None, 'balanced'],
    'clf__ccp_alpha': loguniform(1e-5, 2e-2),
}

dt_search = RandomizedSearchCV(
    estimator=dt_pipe,
    param_distributions=dt_param_dist,
    n_iter=60,                 # DT chạy nhanh, nên cho search rộng hơn
    scoring=scoring,
    refit='f1',
    cv=cv_time,
    n_jobs=-1,
    verbose=0,
    random_state=42,
    return_train_score=False,
    error_score='raise',
)

print("\n>>> Tuning Decision Tree...")
t0 = time.time()
dt_search.fit(X_train, y_train)
elapsed_dt = round(time.time() - t0, 2)

print(f"\nThời gian tuning: {elapsed_dt}s")
print(f"Best params: {dt_search.best_params_}")
print(f"Best CV F1 : {dt_search.best_score_:.4f}")

cv_res_dt = pd.DataFrame(dt_search.cv_results_)
best_idx_dt = dt_search.best_index_
print("\nCV Mean Scores (best params):")
for m in ['f1', 'roc_auc', 'precision', 'recall', 'accuracy']:
    print(f"  {m:12s}: {cv_res_dt.loc[best_idx_dt, f'mean_test_{m}']:.4f}")


In [ ]:
# -------------------------------------------------------
# Threshold tuning bằng OOF từ time folds
# -------------------------------------------------------
# Không dùng cross_val_predict vì expanding CV không tạo partition đầy đủ.
# Fit thủ công theo từng fold để lấy OOF probabilities.

print("\n>>> Threshold tuning – Decision Tree (OOF theo time folds)...")

oof_proba_dt = np.full(len(X_train), np.nan)

for fold_id, (tr_idx, va_idx) in enumerate(cv_time, 1):
    est = clone(dt_search.best_estimator_)
    est.fit(X_train.iloc[tr_idx], y_train[tr_idx])
    oof_proba_dt[va_idx] = est.predict_proba(X_train.iloc[va_idx])[:, 1]
    print(f"  Fold {fold_id}: stored OOF for {len(va_idx):,} rows")

mask_oof = ~np.isnan(oof_proba_dt)

dt_best_thr, dt_oof_f1 = find_best_threshold_f1(
    y_train[mask_oof],
    oof_proba_dt[mask_oof]
)

print(f"Best OOF threshold (F1-max): {dt_best_thr:.4f}")
print(f"OOF Train F1 tại threshold : {dt_oof_f1:.4f}")

# Vẽ F1 theo threshold
prec_dt, rec_dt, thr_dt = precision_recall_curve(y_train[mask_oof], oof_proba_dt[mask_oof])
f1s_dt = 2 * prec_dt[:-1] * rec_dt[:-1] / (prec_dt[:-1] + rec_dt[:-1] + 1e-12)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(thr_dt, f1s_dt, color='#1F77B4', lw=1.5, label='F1 (OOF train)')
ax.plot(thr_dt, prec_dt[:-1], color='#2CA02C', lw=1, linestyle='--', label='Precision')
ax.plot(thr_dt, rec_dt[:-1], color='#D62728', lw=1, linestyle='--', label='Recall')
ax.axvline(dt_best_thr, color='black', linestyle=':', lw=1.5, label=f'Best thr = {dt_best_thr:.3f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Threshold Tuning – Decision Tree (OOF Time CV)')
ax.legend()
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

# -------------------------------------------------------
# Đánh giá trên test
# -------------------------------------------------------

result_dt = evaluate_on_test(
    estimator=dt_search.best_estimator_,
    X_test=X_test,
    y_test=y_test,
    threshold=dt_best_thr,
    model_name='Decision Tree',
)

result_dt['train_time_s'] = elapsed_dt
result_dt['oof_f1'] = round(dt_oof_f1, 4)

## 4.4 Random Forest

In [ ]:
# ============================================================
# Model 4: RANDOM FOREST
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.base import clone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

print("\n" + "=" * 62)
print("  MODEL 4 – RANDOM FOREST")
print("=" * 62)

In [ ]:
# -------------------------------------------------------
# Time-based CV trong train
# -------------------------------------------------------
if 'cv_time' not in globals():
    train_year = df.loc[mask_train, 'yearReport'].reset_index(drop=True)

    valid_years = [2017, 2018, 2019, 2020, 2021, 2022]
    cv_time = []

    for val_year in valid_years:
        tr_idx = np.flatnonzero(train_year < val_year)
        va_idx = np.flatnonzero(train_year == val_year)
        if len(tr_idx) > 0 and len(va_idx) > 0:
            cv_time.append((tr_idx, va_idx))

print("Time folds:")
for i, (tr_idx, va_idx) in enumerate(cv_time, 1):
    print(f"  Fold {i}: train={len(tr_idx):,} | val={len(va_idx):,} | val_year={train_year.iloc[va_idx].iloc[0]}")


In [ ]:
# -------------------------------------------------------
# Pipeline + RandomizedSearchCV
# -------------------------------------------------------
# RF chạy chậm nếu search quá rộng.
# Ở đây giữ search space gọn nhưng vẫn đủ mạnh:
# - Không search bootstrap=False
# - Không search criterion='log_loss'
# - Không search cây quá sâu hoặc quá nhiều cây
# - Không search các tham số pruning ít hiệu quả cho RF

rf_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', RandomForestClassifier(
        n_estimators=300,
        criterion='gini',
        bootstrap=True,
        random_state=42,
        n_jobs=1   # tránh nested parallelism vì search đã dùng n_jobs=-1
    )),
])

rf_param_dist = {
    'clf__n_estimators': [200, 300, 400, 600],
    'clf__max_depth': [6, 8, 10, 12, 14, 16],
    'clf__min_samples_split': [20, 40, 80, 120, 200],
    'clf__min_samples_leaf': [8, 12, 20, 40, 60],
    'clf__max_features': ['sqrt', 0.5, 0.7],
    'clf__class_weight': [
        None,
        'balanced',
        {0: 1, 1: 1.25},
        {0: 1, 1: 1.5},
        {0: 1, 1: 2},
    ],
    'clf__max_samples': [0.7, 0.85, None],
}

rf_search = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=rf_param_dist,
    n_iter=24,               # gọn hơn DT vì RF fit rất lâu
    scoring=scoring,
    refit='f1',              # vẫn ưu tiên F1
    cv=cv_time,
    n_jobs=-1,
    verbose=0,
    random_state=42,
    return_train_score=False,
    error_score='raise',
)

print("\n>>> Tuning Random Forest...")
t0 = time.time()
rf_search.fit(X_train, y_train)
elapsed_rf = round(time.time() - t0, 2)

print(f"\nThời gian tuning: {elapsed_rf}s")
print(f"Best params: {rf_search.best_params_}")
print(f"Best CV F1 : {rf_search.best_score_:.4f}")

cv_res_rf = pd.DataFrame(rf_search.cv_results_)
best_idx_rf = rf_search.best_index_
print("\nCV Mean Scores (best params):")
for m in ['f1', 'roc_auc', 'precision', 'recall', 'accuracy']:
    print(f"  {m:12s}: {cv_res_rf.loc[best_idx_rf, f'mean_test_{m}']:.4f}")

In [ ]:
# -------------------------------------------------------
# Threshold tuning bằng OOF từ time folds
# -------------------------------------------------------
# RF có predict_proba nên vẫn tune threshold theo F1 như DT.

print("\n>>> Threshold tuning – Random Forest (OOF theo time folds)...")

oof_proba_rf = np.full(len(X_train), np.nan)

for fold_id, (tr_idx, va_idx) in enumerate(cv_time, 1):
    est = clone(rf_search.best_estimator_)
    est.fit(X_train.iloc[tr_idx], y_train[tr_idx])
    oof_proba_rf[va_idx] = est.predict_proba(X_train.iloc[va_idx])[:, 1]
    print(f"  Fold {fold_id}: stored OOF for {len(va_idx):,} rows")

mask_oof = ~np.isnan(oof_proba_rf)

rf_best_thr, rf_oof_f1 = find_best_threshold_f1(
    y_train[mask_oof],
    oof_proba_rf[mask_oof]
)

print(f"Best OOF threshold (F1-max): {rf_best_thr:.4f}")
print(f"OOF Train F1 tại threshold : {rf_oof_f1:.4f}")

# Vẽ F1 theo threshold
prec_rf, rec_rf, thr_rf = precision_recall_curve(
    y_train[mask_oof],
    oof_proba_rf[mask_oof]
)
f1s_rf = 2 * prec_rf[:-1] * rec_rf[:-1] / (prec_rf[:-1] + rec_rf[:-1] + 1e-12)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(thr_rf, f1s_rf, color='#2C3E50', lw=1.5, label='F1 (OOF train)')
ax.plot(thr_rf, prec_rf[:-1], color='#27AE60', lw=1, linestyle='--', label='Precision')
ax.plot(thr_rf, rec_rf[:-1], color='#E74C3C', lw=1, linestyle='--', label='Recall')
ax.axvline(rf_best_thr, color='black', linestyle=':', lw=1.5,
           label=f'Best thr = {rf_best_thr:.3f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Threshold Tuning – Random Forest (OOF Time CV)')
ax.legend()
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# -------------------------------------------------------
# Đánh giá trên test
# -------------------------------------------------------

result_rf = evaluate_on_test(
    estimator=rf_search.best_estimator_,
    X_test=X_test,
    y_test=y_test,
    threshold=rf_best_thr,
    model_name='Random Forest',
)

result_rf['train_time_s'] = elapsed_rf
result_rf['oof_f1'] = round(rf_oof_f1, 4)

## 4.5 XGBoost

In [ ]:
# ============================================================
# Model 5: XGBOOST
# ============================================================

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.base import clone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

print("\n" + "=" * 62)
print("  MODEL 5 - XGBOOST")
print("=" * 62)

In [ ]:
# -------------------------------------------------------
# Time-based CV trong train
# -------------------------------------------------------
# Moi model tu tao cv_time de co the chay doc lap.

train_year = df.loc[mask_train, 'yearReport'].reset_index(drop=True)

if 'cv_time' not in globals():
    valid_years = [2017, 2018, 2019, 2020, 2021, 2022]
    cv_time = []

    for val_year in valid_years:
        tr_idx = np.flatnonzero(train_year < val_year)
        va_idx = np.flatnonzero(train_year == val_year)
        if len(tr_idx) > 0 and len(va_idx) > 0:
            cv_time.append((tr_idx, va_idx))

print("Time folds:")
for i, (tr_idx, va_idx) in enumerate(cv_time, 1):
    print(f"  Fold {i}: train={len(tr_idx):,} | val={len(va_idx):,} | val_year={train_year.iloc[va_idx].iloc[0]}")


In [ ]:
# -------------------------------------------------------
# Pipeline + RandomizedSearchCV
# -------------------------------------------------------
# XGBoost la model cay manh nhat trong nhom tree-based.
# Dung tree_method='hist' de tang toc.
# Khong dung SMOTE; uu tien scale_pos_weight + threshold tuning.
# Search space tap trung vao cac tham so tac dong manh nhat.

xgb_spw_base = round((y_train == 0).sum() / max((y_train == 1).sum(), 1), 3)
xgb_scale_pos_weight_grid = sorted(set([
    1.0,
    max(1.0, xgb_spw_base),
    1.15,
    1.3,
    1.5,
]))

xgb_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        tree_method='hist',
        random_state=42,
        n_jobs=1,        # tranh nested parallelism
        verbosity=0
    )),
])

xgb_param_dist = {
    'clf__n_estimators': [200, 350, 500, 700, 900],
    'clf__learning_rate': [0.02, 0.03, 0.05, 0.08, 0.1],
    'clf__max_depth': [3, 4, 5, 6],
    'clf__min_child_weight': [1, 3, 5, 7, 10],
    'clf__subsample': [0.7, 0.85, 1.0],
    'clf__colsample_bytree': [0.6, 0.8, 1.0],
    'clf__gamma': [0, 0.5, 1.0, 2.0],
    'clf__reg_alpha': [0, 1e-3, 1e-2, 0.1, 1.0],
    'clf__reg_lambda': [1, 2, 5, 10],
    'clf__scale_pos_weight': xgb_scale_pos_weight_grid,
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_pipe,
    param_distributions=xgb_param_dist,
    n_iter=40,                 # co the chay lau hon RF mot chut
    scoring=scoring,
    refit='f1',                # uu tien F1
    cv=cv_time,
    n_jobs=-1,
    verbose=0,
    random_state=42,
    return_train_score=False,
    error_score='raise',
)

print("\n>>> Tuning XGBoost...")
t0 = time.time()

total_fit_xgb = xgb_search.n_iter * len(cv_time)
with tqdm_joblib(tqdm(desc="XGB / XGBoost", total=total_fit_xgb)):
    xgb_search.fit(X_train, y_train)

elapsed_xgb = round(time.time() - t0, 2)

print(f"\nThoi gian tuning: {elapsed_xgb}s")
print(f"Best params: {xgb_search.best_params_}")
print(f"Best CV F1 : {xgb_search.best_score_:.4f}")

cv_res_xgb = pd.DataFrame(xgb_search.cv_results_)
best_idx_xgb = xgb_search.best_index_
print("\nCV Mean Scores (best params):")
for m in ['f1', 'roc_auc', 'precision', 'recall', 'accuracy']:
    print(f"  {m:12s}: {cv_res_xgb.loc[best_idx_xgb, f'mean_test_{m}']:.4f}")

In [ ]:
# -------------------------------------------------------
# Threshold tuning bang OOF tu time folds
# -------------------------------------------------------
# Khong dung cross_val_predict vi time-fold nay khong phai partition day du.

print("\n>>> Threshold tuning - XGBoost (OOF theo time folds)...")

oof_proba_xgb = np.full(len(X_train), np.nan)

for fold_id, (tr_idx, va_idx) in enumerate(cv_time, 1):
    est = clone(xgb_search.best_estimator_)
    est.fit(X_train.iloc[tr_idx], y_train[tr_idx])
    oof_proba_xgb[va_idx] = est.predict_proba(X_train.iloc[va_idx])[:, 1]
    print(f"  Fold {fold_id}: stored OOF for {len(va_idx):,} rows")

mask_oof = ~np.isnan(oof_proba_xgb)

xgb_best_thr, xgb_oof_f1 = find_best_threshold_f1(
    y_train[mask_oof],
    oof_proba_xgb[mask_oof]
)

print(f"Best OOF threshold (F1-max): {xgb_best_thr:.4f}")
print(f"OOF Train F1 tai threshold : {xgb_oof_f1:.4f}")

# Ve F1 theo threshold
prec_xgb, rec_xgb, thr_xgb = precision_recall_curve(
    y_train[mask_oof],
    oof_proba_xgb[mask_oof]
)
f1s_xgb = 2 * prec_xgb[:-1] * rec_xgb[:-1] / (prec_xgb[:-1] + rec_xgb[:-1] + 1e-12)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(thr_xgb, f1s_xgb, color='#C0392B', lw=1.5, label='F1 (OOF train)')
ax.plot(thr_xgb, prec_xgb[:-1], color='#27AE60', lw=1, linestyle='--', label='Precision')
ax.plot(thr_xgb, rec_xgb[:-1], color='#2C3E50', lw=1, linestyle='--', label='Recall')
ax.axvline(xgb_best_thr, color='black', linestyle=':', lw=1.5,
           label=f'Best thr = {xgb_best_thr:.3f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Threshold Tuning - XGBoost (OOF Time CV)')
ax.legend()
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# -------------------------------------------------------
# Danh gia tren test
# -------------------------------------------------------
# Ham evaluate_on_test da tu ve ROC + Confusion Matrix.

result_xgb = evaluate_on_test(
    estimator=xgb_search.best_estimator_,
    X_test=X_test,
    y_test=y_test,
    threshold=xgb_best_thr,
    model_name='XGBoost',
)

result_xgb['train_time_s'] = elapsed_xgb
result_xgb['oof_f1'] = round(xgb_oof_f1, 4)

## 4.6 SVM

In [ ]:
# ============================================================
# Model 6: SVM
# LinearSVC + CalibratedClassifierCV
# ============================================================

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.base import clone
from scipy.stats import loguniform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

print("\n" + "=" * 62)
print("  MODEL 6 - SVM (LinearSVC + Calibration)")
print("=" * 62)

# -------------------------------------------------------
# Time-based CV trong train
# -------------------------------------------------------

train_year = df.loc[mask_train, 'yearReport'].reset_index(drop=True)

if 'cv_time' not in globals():
    valid_years = [2017, 2018, 2019, 2020, 2021, 2022]
    cv_time = []

    for val_year in valid_years:
        tr_idx = np.flatnonzero(train_year < val_year)
        va_idx = np.flatnonzero(train_year == val_year)
        if len(tr_idx) > 0 and len(va_idx) > 0:
            cv_time.append((tr_idx, va_idx))

print("Time folds:")
for i, (tr_idx, va_idx) in enumerate(cv_time, 1):
    print(f"  Fold {i}: train={len(tr_idx):,} | val={len(va_idx):,} | val_year={train_year.iloc[va_idx].iloc[0]}")


In [ ]:
# -------------------------------------------------------
# Pipeline + RandomizedSearchCV
# -------------------------------------------------------
# Dung LinearSVC de scale tot hon voi 12k+ samples.
# Dung CalibratedClassifierCV(method='sigmoid') de co predict_proba
# cho ROC, threshold tuning va evaluate_on_test.

svm_class_weight_grid = [
    None,
    'balanced',
    {0: 1, 1: 1.25},
    {0: 1, 1: 1.5},
    {0: 1, 1: 2},
]

svm_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('clf', CalibratedClassifierCV(
        estimator=LinearSVC(
            dual='auto',
            random_state=42
        ),
        method='sigmoid',
        cv=3,              # calibration noi bo
        ensemble=False     # nhe hon, gan logic probability=True cua SVC
    )),
])

svm_param_dist = {
    'clf__estimator__C': loguniform(1e-3, 50),
    'clf__estimator__class_weight': svm_class_weight_grid,
    'clf__estimator__max_iter': [3000, 5000, 8000],
}

svm_search = RandomizedSearchCV(
    estimator=svm_pipe,
    param_distributions=svm_param_dist,
    n_iter=20,
    scoring=scoring,
    refit='f1',
    cv=cv_time,
    n_jobs=-1,
    verbose=0,
    random_state=42,
    return_train_score=False,
    error_score='raise',
)

print("\n>>> Tuning SVM...")
t0 = time.time()

total_fit_svm = svm_search.n_iter * len(cv_time)
with tqdm_joblib(tqdm(desc="SVM / LinearSVC + CalibratedCV", total=total_fit_svm)):
    svm_search.fit(X_train, y_train)

elapsed_svm = round(time.time() - t0, 2)

print(f"\nThoi gian tuning: {elapsed_svm}s")
print(f"Best params: {svm_search.best_params_}")
print(f"Best CV F1 : {svm_search.best_score_:.4f}")

cv_res_svm = pd.DataFrame(svm_search.cv_results_)
best_idx_svm = svm_search.best_index_
print("\nCV Mean Scores (best params):")
for m in ['f1', 'roc_auc', 'precision', 'recall', 'accuracy']:
    print(f"  {m:12s}: {cv_res_svm.loc[best_idx_svm, f'mean_test_{m}']:.4f}")

In [ ]:
# -------------------------------------------------------
# Threshold tuning bang OOF tu time folds
# -------------------------------------------------------

print("\n>>> Threshold tuning - SVM (OOF theo time folds)...")

oof_proba_svm = np.full(len(X_train), np.nan)

for fold_id, (tr_idx, va_idx) in enumerate(cv_time, 1):
    est = clone(svm_search.best_estimator_)
    est.fit(X_train.iloc[tr_idx], y_train[tr_idx])
    oof_proba_svm[va_idx] = est.predict_proba(X_train.iloc[va_idx])[:, 1]
    print(f"  Fold {fold_id}: stored OOF for {len(va_idx):,} rows")

mask_oof = ~np.isnan(oof_proba_svm)

svm_best_thr, svm_oof_f1 = find_best_threshold_f1(
    y_train[mask_oof],
    oof_proba_svm[mask_oof]
)

print(f"Best OOF threshold (F1-max): {svm_best_thr:.4f}")
print(f"OOF Train F1 tai threshold : {svm_oof_f1:.4f}")

# Ve F1 theo threshold
prec_svm, rec_svm, thr_svm = precision_recall_curve(
    y_train[mask_oof],
    oof_proba_svm[mask_oof]
)
f1s_svm = 2 * prec_svm[:-1] * rec_svm[:-1] / (prec_svm[:-1] + rec_svm[:-1] + 1e-12)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(thr_svm, f1s_svm, color='#8E44AD', lw=1.5, label='F1 (OOF train)')
ax.plot(thr_svm, prec_svm[:-1], color='#27AE60', lw=1, linestyle='--', label='Precision')
ax.plot(thr_svm, rec_svm[:-1], color='#E74C3C', lw=1, linestyle='--', label='Recall')
ax.axvline(svm_best_thr, color='black', linestyle=':', lw=1.5,
           label=f'Best thr = {svm_best_thr:.3f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title('Threshold Tuning - SVM (OOF Time CV)')
ax.legend()
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()


In [ ]:
# -------------------------------------------------------
# Danh gia tren test
# -------------------------------------------------------

result_svm = evaluate_on_test(
    estimator=svm_search.best_estimator_,
    X_test=X_test,
    y_test=y_test,
    threshold=svm_best_thr,
    model_name='SVM (LinearSVC + CalibratedCV)',
)

result_svm['train_time_s'] = elapsed_svm
result_svm['oof_f1'] = round(svm_oof_f1, 4)

In [ ]:
# ============================================================
# TONG HOP KET QUA 6 MODEL
# ============================================================

print("\n" + "=" * 62)
print("  KET QUA TONG HOP: 6 MODELS")
print("=" * 62)

summary_cols = [
    'model', 'threshold',
    'accuracy', 'precision', 'recall', 'f1',
    'roc_auc', 'pr_auc',
    'oof_f1', 'train_time_s'
]

summary_df = pd.DataFrame([
    result_lg,
    result_ann,
    result_dt,
    result_rf,
    result_svm,
    result_xgb,
])[summary_cols]

summary_df = summary_df.sort_values(
    by=['f1', 'roc_auc', 'pr_auc'],
    ascending=False
).reset_index(drop=True)

print(summary_df.to_string(index=False))
summary_df


### So sánh đường cong ROC của 6 mô hình

In [ ]:
from sklearn.metrics import roc_curve, auc

# Danh sách các model và kết quả tương ứng đã train
models_to_plot = [
    ('Logistic Regression', lg_search.best_estimator_, '#3498DB'),
    ('ANN', ann_search.best_estimator_, '#8E44AD'),
    ('Decision Tree', dt_search.best_estimator_, '#E67E22'),
    ('Random Forest', rf_search.best_estimator_, '#2C3E50'),
    ('XGBoost', xgb_search.best_estimator_, '#E74C3C'),
    ('SVM', svm_search.best_estimator_, '#27AE60')
]

plt.figure(figsize=(10, 8))

for name, model, color in models_to_plot:
    # Lấy xác suất dự báo lớp 1
    y_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, color=color, lw=2,
             label=f'{name} (AUC = {roc_auc:.3f})')

# Vẽ đường baseline (ngẫu nhiên)
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', alpha=0.5)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('So sánh đường cong ROC - 6 Mô hình dự báo KQTC', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=10)
plt.grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('fig_comparison_roc_all_models.png', dpi=150, bbox_inches='tight')
plt.show()

---
# PHẦN 5: SHAP

In [ ]:
!pip install shap

In [ ]:
import shap

# 1. Tự động lấy mô hình có F1 cao nhất từ bảng tổng hợp
best_model_name = summary_df.iloc[0]['model']
print(f"Mô hình được chọn để giải thích: {best_model_name}")

# Ánh xạ tên mô hình sang đối tượng search tương ứng
model_mapping = {
    'Logistic Regression': lg_search,
    'ANN (MLPClassifier)': ann_search,
    'Decision Tree': dt_search,
    'Random Forest': rf_search,
    'XGBoost': xgb_search,
    'SVM (LinearSVC + CalibratedCV)': svm_search
}

best_estimator = model_mapping[best_model_name].best_estimator_

# 2. Chuẩn bị dữ liệu đã qua xử lý
preprocessor_step = best_estimator.named_steps['preprocessor']
X_test_transformed = preprocessor_step.transform(X_test)
feature_names = preprocessor_step.get_feature_names_out()
X_test_disp = pd.DataFrame(X_test_transformed, columns=feature_names)

# 3. Khởi tạo SHAP Explainer
# Lưu ý: TreeExplainer cho các mô hình cây, KernelExplainer hoặc Explainer chung cho các loại khác
clf_model = best_estimator.named_steps['clf']

if best_model_name in ['XGBoost', 'Random Forest', 'Decision Tree']:
    explainer = shap.TreeExplainer(clf_model)
    shap_values = explainer.shap_values(X_test_disp)
else:
    # Sử dụng mẫu nhỏ (100 dòng) cho KernelExplainer để tiết kiệm thời gian nếu không phải cây
    explainer = shap.Explainer(clf_model.predict, shap.maskers.Independent(X_test_disp), seed=42)
    shap_values = explainer(X_test_disp.head(100))

# 4. Trực quan hóa
plt.figure(figsize=(10, 8))
if isinstance(shap_values, list):
    # Đối với một số model output list (như RF), lấy class 1
    shap.summary_plot(shap_values[1], X_test_disp, plot_type="dot", show=False)
else:
    shap.summary_plot(shap_values, X_test_disp, plot_type="dot", show=False)

plt.title(f'SHAP Summary Plot - Tầm quan trọng các biến ({best_model_name})', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('fig_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import shap

# Calculate the global importance (mean absolute SHAP value) and plot as a bar chart
plt.figure(figsize=(10, 8))

# shap_values from the previous cell is used here
# If it is a list (for multi-class or some tree models), we select the class of interest (class 1)
if isinstance(shap_values, list):
    shap.summary_plot(shap_values[1], X_test_disp, plot_type="bar", show=False)
else:
    shap.summary_plot(shap_values, X_test_disp, plot_type="bar", show=False)

plt.title(f'SHAP Global Feature Importance - {best_model_name}', fontsize=14, fontweight='bold')
plt.xlabel('mean(|SHAP value|) (Average impact on model output magnitude)')
plt.tight_layout()
plt.show()